# Pandas Part 2 — Intermediate & Advanced

This notebook builds on **Pandas Part 1** and focuses on the most important intermediate and advanced Pandas tools used in **data analysis, Bioinformatics, Machine Learning, and scientific computing**.

> **Part 2 = Data manipulation, transformation, analysis, reshaping, merging, and ML preparation.**


## Learning Objectives

- Count and inspect unique values.
- Group and aggregate data.
- Use advanced filtering and transformation.
- Handle missing data.
- Merge, join, and concatenate tables.
- Reshape data.
- Work with categorical and datetime data.
- Perform statistical and sequential analysis.
- Prepare features for Machine Learning.
- Optimize memory and build practical workflows.


## 0. Setup

Install with `pip install pandas numpy` and import:


In [1]:
import pandas as pd
import numpy as np
print("Pandas version:", pd.__version__)


Pandas version: 3.0.3


## Reusable Example Dataset
We will use a small synthetic clinical/Bioinformatics-style dataset throughout the notebook.


In [2]:
patients = pd.DataFrame({
    "Patient_ID": ["P001","P002","P003","P004","P005","P006","P007","P008","P009","P010"],
    "Age": [45,62,51,70,38,62,55,70,47,59],
    "Gender": ["Female","Male","Female","Male","Female","Male","Female","Male","Female","Male"],
    "Tumor_Type": ["GBM","GBM","LGG","GBM","LGG","GBM","GBM","LGG","GBM","LGG"],
    "Treatment": ["Surgery","Chemotherapy","Radiation","Surgery","Radiation","Chemotherapy","Surgery","Radiation","Chemotherapy","Surgery"],
    "Expression": [8.2,6.5,4.1,9.0,3.8,7.2,8.7,4.5,6.9,5.2],
    "Survival_Days": [520,310,780,240,920,410,350,860,480,690]
})
patients


,Patient_ID,Age,Gender,Tumor_Type,Treatment,Expression,Survival_Days
0,P001,45,Female,GBM,Surgery,8.2,520
1,P002,62,Male,GBM,Chemotherapy,6.5,310
2,P003,51,Female,LGG,Radiation,4.1,780
3,P004,70,Male,GBM,Surgery,9.0,240
4,P005,38,Female,LGG,Radiation,3.8,920
5,P006,62,Male,GBM,Chemotherapy,7.2,410
6,P007,55,Female,GBM,Surgery,8.7,350
7,P008,70,Male,LGG,Radiation,4.5,860
8,P009,47,Female,GBM,Chemotherapy,6.9,480
9,P010,59,Male,LGG,Surgery,5.2,690


# 1. Counting and Uniqueness
`value_counts()`, `unique()`, and `nunique()` are essential EDA tools.


In [3]:
print(patients["Tumor_Type"].value_counts())
print(patients["Tumor_Type"].value_counts(normalize=True))
print(patients["Treatment"].unique())
print("Unique tumor types:", patients["Tumor_Type"].nunique())


Tumor_Type
GBM    6
LGG    4
Name: count, dtype: int64
Tumor_Type
GBM    0.6
LGG    0.4
Name: proportion, dtype: float64
<StringArray>
['Surgery', 'Chemotherapy', 'Radiation']
Length: 3, dtype: str
Unique tumor types: 2


# 2. GroupBy and Aggregation
`groupby()` follows Split → Apply → Combine. Use `agg()` for summaries and `transform()` when the output must retain the original row count.


In [4]:
print(patients.groupby("Tumor_Type")["Survival_Days"].mean())
summary = patients.groupby("Tumor_Type")["Survival_Days"].agg(["count","mean","median","min","max"])
print(summary)


Tumor_Type
GBM    385.0
LGG    812.5
Name: Survival_Days, dtype: float64
            count   mean  median  min  max
Tumor_Type                                
GBM             6  385.0   380.0  240  520
LGG             4  812.5   820.0  690  920


In [5]:
group_summary = patients.groupby("Tumor_Type").agg({"Age":["mean","median"],"Expression":["mean","max"],"Survival_Days":["mean","min","max"]})
print(group_summary)


                  Age        Expression      Survival_Days          
                 mean median       mean  max          mean  min  max
Tumor_Type                                                          
GBM         56.833333   58.5       7.75  9.0         385.0  240  520
LGG         54.500000   55.0       4.40  5.2         812.5  690  920


In [6]:
patients["Tumor_Mean_Expression"] = patients.groupby("Tumor_Type")["Expression"].transform("mean")
patients[["Patient_ID","Tumor_Type","Expression","Tumor_Mean_Expression"]]


,Patient_ID,Tumor_Type,Expression,Tumor_Mean_Expression
0,P001,GBM,8.2,7.75
1,P002,GBM,6.5,7.75
2,P003,LGG,4.1,4.40
3,P004,GBM,9.0,7.75
4,P005,LGG,3.8,4.40
5,P006,GBM,7.2,7.75
6,P007,GBM,8.7,7.75
7,P008,LGG,4.5,4.40
8,P009,GBM,6.9,7.75
9,P010,LGG,5.2,4.40


In [7]:
def age_group(age):
    if age < 50: return "Young"
    elif age < 65: return "Middle"
    return "Older"
patients["Age_Group"] = patients["Age"].apply(age_group)
patients[["Age","Age_Group"]]


,Age,Age_Group
0,45,Young
1,62,Middle
2,51,Middle
3,70,Older
4,38,Young
5,62,Middle
6,55,Middle
7,70,Older
8,47,Young
9,59,Middle


In [8]:
gender_mapping = {"Female":0,"Male":1}
patients["Gender_Encoded"] = patients["Gender"].map(gender_mapping)
patients[["Gender","Gender_Encoded"]]


,Gender,Gender_Encoded
0,Female,0
1,Male,1
2,Female,0
3,Male,1
4,Female,0
5,Male,1
6,Female,0
7,Male,1
8,Female,0
9,Male,1


# 3. Advanced Filtering
Use `isin()`, `between()`, `query()`, `where()`, and `mask()` for expressive filtering.


In [9]:
print(patients[patients["Treatment"].isin(["Surgery","Radiation"])])
print(patients[patients["Age"].between(50,65)])
print(patients.query("Age >= 60 and Survival_Days < 500"))


  Patient_ID  Age  Gender Tumor_Type  Treatment  Expression  Survival_Days  \
0       P001   45  Female        GBM    Surgery         8.2            520   
2       P003   51  Female        LGG  Radiation         4.1            780   
3       P004   70    Male        GBM    Surgery         9.0            240   
4       P005   38  Female        LGG  Radiation         3.8            920   
6       P007   55  Female        GBM    Surgery         8.7            350   
7       P008   70    Male        LGG  Radiation         4.5            860   
9       P010   59    Male        LGG    Surgery         5.2            690   

   Tumor_Mean_Expression Age_Group  Gender_Encoded  
0                   7.75     Young               0  
2                   4.40    Middle               0  
3                   7.75     Older               1  
4                   4.40     Young               0  
6                   7.75    Middle               0  
7                   4.40     Older               1  
9   

In [10]:
patients["High_Expression"] = patients["Expression"].where(patients["Expression"] >= 7)
patients["Masked_Expression"] = patients["Expression"].mask(patients["Expression"] < 5)
patients[["Expression","High_Expression","Masked_Expression"]]


,Expression,High_Expression,Masked_Expression
0,8.2,8.2,8.2
1,6.5,NaN,6.5
2,4.1,NaN,NaN
3,9.0,9.0,9.0
4,3.8,NaN,NaN
5,7.2,7.2,7.2
6,8.7,8.7,8.7
7,4.5,NaN,NaN
8,6.9,NaN,6.9
9,5.2,NaN,5.2


# 4. Data Transformation
Important tools: `replace()`, `assign()`, `clip()`, `rank()`, `nlargest()`, `nsmallest()`, `idxmax()`, `idxmin()`.


In [11]:
demo = patients.copy()
demo["Treatment"] = demo["Treatment"].replace({"Chemotherapy":"Chemo"})
print(demo[["Patient_ID","Treatment"]])
assigned = patients.assign(Survival_Years=patients["Survival_Days"]/365.25, Expression_Log=np.log1p(patients["Expression"]))
print(assigned[["Patient_ID","Survival_Years","Expression_Log"]])


  Patient_ID  Treatment
0       P001    Surgery
1       P002      Chemo
2       P003  Radiation
3       P004    Surgery
4       P005  Radiation
5       P006      Chemo
6       P007    Surgery
7       P008  Radiation
8       P009      Chemo
9       P010    Surgery
  Patient_ID  Survival_Years  Expression_Log
0       P001        1.423682        2.219203
1       P002        0.848734        2.014903
2       P003        2.135524        1.629241
3       P004        0.657084        2.302585
4       P005        2.518823        1.568616
5       P006        1.122519        2.104134
6       P007        0.958248        2.272126
7       P008        2.354552        1.704748
8       P009        1.314168        2.066863
9       P010        1.889117        1.824549


In [12]:
patients["Age_Clipped"] = patients["Age"].clip(40,65)
patients["Survival_Rank"] = patients["Survival_Days"].rank(ascending=False, method="dense")
print(patients.nlargest(3,"Survival_Days")[["Patient_ID","Survival_Days"]])
print(patients.nsmallest(3,"Survival_Days")[["Patient_ID","Survival_Days"]])
print("Max index:", patients["Survival_Days"].idxmax(), "Min index:", patients["Survival_Days"].idxmin())


  Patient_ID  Survival_Days
4       P005            920
7       P008            860
2       P003            780
  Patient_ID  Survival_Days
3       P004            240
1       P002            310
6       P007            350
Max index: 4 Min index: 3


# 5. Advanced Missing-Data Handling
Use `dropna()`, `fillna()`, `ffill()`, `bfill()`, `interpolate()`, and `combine_first()` based on the meaning of the missingness.


In [13]:
missing = patients.copy()
missing.loc[2,"Expression"] = np.nan
missing.loc[5,"Expression"] = np.nan
missing.loc[7,"Survival_Days"] = np.nan
print(missing)
print("Rows after dropna:", len(missing.dropna()))
print(missing.dropna(subset=["Survival_Days"]))


  Patient_ID  Age  Gender Tumor_Type     Treatment  Expression  Survival_Days  \
0       P001   45  Female        GBM       Surgery         8.2          520.0   
1       P002   62    Male        GBM  Chemotherapy         6.5          310.0   
2       P003   51  Female        LGG     Radiation         NaN          780.0   
3       P004   70    Male        GBM       Surgery         9.0          240.0   
4       P005   38  Female        LGG     Radiation         3.8          920.0   
5       P006   62    Male        GBM  Chemotherapy         NaN          410.0   
6       P007   55  Female        GBM       Surgery         8.7          350.0   
7       P008   70    Male        LGG     Radiation         4.5            NaN   
8       P009   47  Female        GBM  Chemotherapy         6.9          480.0   
9       P010   59    Male        LGG       Surgery         5.2          690.0   

   Tumor_Mean_Expression Age_Group  Gender_Encoded  High_Expression  \
0                   7.75     Young   

In [15]:
filled = missing.copy()

filled["Expression"] = filled["Expression"].fillna(
    filled["Expression"].median()
)

print(filled)


series = pd.Series([10, np.nan, np.nan, 40, np.nan, 60])

print("ffill:")
print(series.ffill())

print("bfill:")
print(series.bfill())

print("interpolate:")
print(series.interpolate())

  Patient_ID  Age  Gender Tumor_Type     Treatment  Expression  Survival_Days  \
0       P001   45  Female        GBM       Surgery         8.2          520.0   
1       P002   62    Male        GBM  Chemotherapy         6.5          310.0   
2       P003   51  Female        LGG     Radiation         6.7          780.0   
3       P004   70    Male        GBM       Surgery         9.0          240.0   
4       P005   38  Female        LGG     Radiation         3.8          920.0   
5       P006   62    Male        GBM  Chemotherapy         6.7          410.0   
6       P007   55  Female        GBM       Surgery         8.7          350.0   
7       P008   70    Male        LGG     Radiation         4.5            NaN   
8       P009   47  Female        GBM  Chemotherapy         6.9          480.0   
9       P010   59    Male        LGG       Surgery         5.2          690.0   

   Tumor_Mean_Expression Age_Group  Gender_Encoded  High_Expression  \
0                   7.75     Young   

In [16]:
a = pd.Series([10,np.nan,30,np.nan])
b = pd.Series([100,200,300,400])
print(a.combine_first(b))


0     10.0
1    200.0
2     30.0
3    400.0
dtype: float64


# 6. Duplicate Detection
`duplicated()` identifies duplicate rows; `drop_duplicates()` removes them.


In [17]:
duplicates = pd.concat([patients,patients.iloc[[0]]],ignore_index=True)
print(duplicates[duplicates.duplicated()])
print(duplicates.drop_duplicates())
print(duplicates[duplicates.duplicated(subset=["Age","Gender"],keep=False)])


   Patient_ID  Age  Gender Tumor_Type Treatment  Expression  Survival_Days  \
10       P001   45  Female        GBM   Surgery         8.2            520   

    Tumor_Mean_Expression Age_Group  Gender_Encoded  High_Expression  \
10                   7.75     Young               0              8.2   

    Masked_Expression  Age_Clipped  Survival_Rank  
10                8.2           45            5.0  
  Patient_ID  Age  Gender Tumor_Type     Treatment  Expression  Survival_Days  \
0       P001   45  Female        GBM       Surgery         8.2            520   
1       P002   62    Male        GBM  Chemotherapy         6.5            310   
2       P003   51  Female        LGG     Radiation         4.1            780   
3       P004   70    Male        GBM       Surgery         9.0            240   
4       P005   38  Female        LGG     Radiation         3.8            920   
5       P006   62    Male        GBM  Chemotherapy         7.2            410   
6       P007   55  Female  

# 7. Combining Multiple DataFrames
Use `merge()` for relational joins, `join()` when indexes are convenient, and `concat()` for stacking tables.


In [18]:
clinical = patients[["Patient_ID","Age","Gender","Tumor_Type"]].copy()
molecular = pd.DataFrame({"Patient_ID":["P001","P002","P003","P004","P005"],"Gene":["TP53","EGFR","IDH1","TP53","IDH1"],"Expression":[8.2,6.5,4.1,9.0,3.8]})
inner = pd.merge(clinical,molecular,on="Patient_ID",how="inner")
left = pd.merge(clinical,molecular,on="Patient_ID",how="left")
print(inner)
print(left)


  Patient_ID  Age  Gender Tumor_Type  Gene  Expression
0       P001   45  Female        GBM  TP53         8.2
1       P002   62    Male        GBM  EGFR         6.5
2       P003   51  Female        LGG  IDH1         4.1
3       P004   70    Male        GBM  TP53         9.0
4       P005   38  Female        LGG  IDH1         3.8
  Patient_ID  Age  Gender Tumor_Type  Gene  Expression
0       P001   45  Female        GBM  TP53         8.2
1       P002   62    Male        GBM  EGFR         6.5
2       P003   51  Female        LGG  IDH1         4.1
3       P004   70    Male        GBM  TP53         9.0
4       P005   38  Female        LGG  IDH1         3.8
5       P006   62    Male        GBM   NaN         NaN
6       P007   55  Female        GBM   NaN         NaN
7       P008   70    Male        LGG   NaN         NaN
8       P009   47  Female        GBM   NaN         NaN
9       P010   59    Male        LGG   NaN         NaN


In [19]:
molecular_alt = molecular.rename(columns={"Patient_ID":"Sample_ID"})
print(pd.merge(clinical,molecular_alt,left_on="Patient_ID",right_on="Sample_ID",how="left"))
clinical_indexed=clinical.set_index("Patient_ID")
molecular_indexed=molecular.set_index("Patient_ID")
print(clinical_indexed.join(molecular_indexed[["Gene","Expression"]],how="left"))


  Patient_ID  Age  Gender Tumor_Type Sample_ID  Gene  Expression
0       P001   45  Female        GBM      P001  TP53         8.2
1       P002   62    Male        GBM      P002  EGFR         6.5
2       P003   51  Female        LGG      P003  IDH1         4.1
3       P004   70    Male        GBM      P004  TP53         9.0
4       P005   38  Female        LGG      P005  IDH1         3.8
5       P006   62    Male        GBM       NaN   NaN         NaN
6       P007   55  Female        GBM       NaN   NaN         NaN
7       P008   70    Male        LGG       NaN   NaN         NaN
8       P009   47  Female        GBM       NaN   NaN         NaN
9       P010   59    Male        LGG       NaN   NaN         NaN
            Age  Gender Tumor_Type  Gene  Expression
Patient_ID                                          
P001         45  Female        GBM  TP53         8.2
P002         62    Male        GBM  EGFR         6.5
P003         51  Female        LGG  IDH1         4.1
P004         70    M

In [20]:
part1=patients.iloc[:5]
part2=patients.iloc[5:]
print(pd.concat([part1,part2],ignore_index=True).head())
print(pd.concat([patients[["Patient_ID","Age"]],patients[["Treatment","Expression"]]],axis=1).head())


  Patient_ID  Age  Gender Tumor_Type     Treatment  Expression  Survival_Days  \
0       P001   45  Female        GBM       Surgery         8.2            520   
1       P002   62    Male        GBM  Chemotherapy         6.5            310   
2       P003   51  Female        LGG     Radiation         4.1            780   
3       P004   70    Male        GBM       Surgery         9.0            240   
4       P005   38  Female        LGG     Radiation         3.8            920   

   Tumor_Mean_Expression Age_Group  Gender_Encoded  High_Expression  \
0                   7.75     Young               0              8.2   
1                   7.75    Middle               1              NaN   
2                   4.40    Middle               0              NaN   
3                   7.75     Older               1              9.0   
4                   4.40     Young               0              NaN   

   Masked_Expression  Age_Clipped  Survival_Rank  
0                8.2           45  

# 8. Reshaping Data
Use `melt()` for Wide → Long, `pivot()` for Long → Wide, `pivot_table()` when aggregation is needed, `crosstab()` for frequency tables, and `explode()` for list-like columns.


In [21]:
expression_wide=pd.DataFrame({"Patient_ID":["P001","P002","P003"],"TP53":[8.2,6.5,7.1],"EGFR":[4.2,8.1,5.0],"IDH1":[2.1,3.5,7.8]})
expression_long=expression_wide.melt(id_vars="Patient_ID",var_name="Gene",value_name="Expression")
print(expression_long)
print(expression_long.pivot(index="Patient_ID",columns="Gene",values="Expression"))


  Patient_ID  Gene  Expression
0       P001  TP53         8.2
1       P002  TP53         6.5
2       P003  TP53         7.1
3       P001  EGFR         4.2
4       P002  EGFR         8.1
5       P003  EGFR         5.0
6       P001  IDH1         2.1
7       P002  IDH1         3.5
8       P003  IDH1         7.8
Gene        EGFR  IDH1  TP53
Patient_ID                  
P001         4.2   2.1   8.2
P002         8.1   3.5   6.5
P003         5.0   7.8   7.1


In [22]:
dup=pd.concat([expression_long,expression_long.iloc[[0]]],ignore_index=True)
print(dup.pivot_table(index="Patient_ID",columns="Gene",values="Expression",aggfunc="mean"))
print(pd.crosstab(patients["Tumor_Type"],patients["Treatment"]))
print(pd.crosstab(patients["Tumor_Type"],patients["Treatment"],normalize="index"))


Gene        EGFR  IDH1  TP53
Patient_ID                  
P001         4.2   2.1   8.2
P002         8.1   3.5   6.5
P003         5.0   7.8   7.1
Treatment   Chemotherapy  Radiation  Surgery
Tumor_Type                                  
GBM                    3          0        3
LGG                    0          3        1
Treatment   Chemotherapy  Radiation  Surgery
Tumor_Type                                  
GBM                  0.5       0.00     0.50
LGG                  0.0       0.75     0.25


In [23]:
gene_lists=pd.DataFrame({"Patient_ID":["P001","P002","P003"],"Genes":[["TP53","EGFR"],["IDH1","ATRX","TP53"],["EGFR"]]})
print(gene_lists.explode("Genes"))


  Patient_ID Genes
0       P001  TP53
0       P001  EGFR
1       P002  IDH1
1       P002  ATRX
1       P002  TP53
2       P003  EGFR


# 9. Binning and Categorical Data
`cut()` uses defined intervals; `qcut()` creates quantile-based bins; categorical dtype can save memory and clarify semantics.


In [24]:
patients["Age_Bin"]=pd.cut(patients["Age"],bins=[0,49,64,100],labels=["Young","Middle","Older"])
patients["Survival_Quantile"]=pd.qcut(patients["Survival_Days"],q=3,labels=["Low","Medium","High"])
print(patients[["Age","Age_Bin","Survival_Days","Survival_Quantile"]])
cat=patients.copy()
for c in ["Gender","Tumor_Type","Treatment"]: cat[c]=cat[c].astype("category")
print(cat.dtypes)


   Age Age_Bin  Survival_Days Survival_Quantile
0   45   Young            520            Medium
1   62  Middle            310               Low
2   51  Middle            780              High
3   70   Older            240               Low
4   38   Young            920              High
5   62  Middle            410               Low
6   55  Middle            350               Low
7   70   Older            860              High
8   47   Young            480            Medium
9   59  Middle            690            Medium
Patient_ID                    str
Age                         int64
Gender                   category
Tumor_Type               category
Treatment                category
Expression                float64
Survival_Days               int64
Tumor_Mean_Expression     float64
Age_Group                     str
Gender_Encoded              int64
High_Expression           float64
Masked_Expression         float64
Age_Clipped                 int64
Survival_Rank             floa

# 10. Statistical Analysis
`corr()` calculates correlation and `cov()` calculates covariance. Correlation measures association, not causation.


In [25]:
numeric=patients[["Age","Expression","Survival_Days"]]
print(numeric.corr())
print(numeric.cov())
values=pd.Series([10,20,15,30,25])
print("cumsum:
",values.cumsum())
print("cummin:
",values.cummin())
print("cummax:
",values.cummax())


SyntaxError: unterminated string literal (detected at line 5) (2618525797.py, line 5)

# 11. Sequential and Time-Series Operations
`shift()`, `diff()`, `pct_change()`, `rolling()`, and `expanding()` are useful for ordered observations.


In [ ]:
patients["Previous_Survival"]=patients["Survival_Days"].shift(1)
patients["Survival_Difference"]=patients["Survival_Days"].diff()
patients["Survival_Pct_Change"]=patients["Survival_Days"].pct_change()
patients["Rolling_Mean_Expression"]=patients["Expression"].rolling(3).mean()
patients["Expanding_Mean_Expression"]=patients["Expression"].expanding().mean()
patients[["Patient_ID","Survival_Days","Previous_Survival","Survival_Difference","Survival_Pct_Change"]]


# 12. Date and Time Data
Use `pd.to_datetime()` and the `.dt` accessor to extract date components and calculate durations.


In [ ]:
dates=pd.DataFrame({"Patient_ID":["P001","P002","P003","P004"],"Diagnosis_Date":["2024-01-15","2024-03-20","2025-06-10","2025-09-01"]})
dates["Diagnosis_Date"]=pd.to_datetime(dates["Diagnosis_Date"])
dates["Year"]=dates["Diagnosis_Date"].dt.year
dates["Month"]=dates["Diagnosis_Date"].dt.month
dates["Day"]=dates["Diagnosis_Date"].dt.day
dates["Day_of_Week"]=dates["Diagnosis_Date"].dt.dayofweek
dates["Follow_Up_Date"]=dates["Diagnosis_Date"]+pd.to_timedelta(180,unit="D")
dates["Follow_Up_Days"]=(dates["Follow_Up_Date"]-dates["Diagnosis_Date"]).dt.days
dates


# 13. Index Management
`set_index()` creates an index from columns; `reset_index()` returns index levels to columns. MultiIndex supports hierarchical indexing.


In [ ]:
indexed=patients.set_index("Patient_ID")
print(indexed.head())
print(indexed.reset_index().head())
multi=patients.set_index(["Tumor_Type","Treatment"])
print(multi.head())
print(multi.reset_index().head())


# 14. Preparing Data for Machine Learning
Common Pandas preprocessing tasks include encoding categories, selecting numerical columns, and feature engineering.


In [ ]:
encoded=pd.get_dummies(patients[["Gender","Tumor_Type","Treatment"]],dtype=int)
print(encoded)
codes,categories=pd.factorize(patients["Tumor_Type"])
print("Codes:",codes)
print("Categories:",categories)
print(patients.select_dtypes(include="number").head())


# 15. Advanced String Operations
The `.str` accessor provides vectorized string operations such as `contains()`, `startswith()`, `endswith()`, `split()`, `extract()`, and `replace()`.


In [ ]:
samples=pd.DataFrame({"Sample_ID":["GBM_001_T1","GBM_002_T2","LGG_003_T1","GBM_004_T2"],"Description":["primary tumor tissue","recurrent tumor tissue","primary tumor tissue","normal adjacent tissue"]})
print(samples[samples["Sample_ID"].str.contains("GBM")])
samples["Tumor_Class"]=samples["Sample_ID"].str.split("_").str[0]
samples["Sample_Number"]=samples["Sample_ID"].str.split("_").str[1]
samples["Numeric_ID"]=samples["Sample_ID"].str.extract(r"_(\d+)_")
samples["Clean_Description"]=samples["Description"].str.replace("tissue","sample",regex=False)
samples


# 16. Advanced DataFrame Workflow
Method chaining can make multi-step processing readable.


In [ ]:
workflow=(patients.query("Age >= 45").assign(Survival_Years=lambda df:df["Survival_Days"]/365.25,Expression_Log=lambda df:np.log1p(df["Expression"])).sort_values("Survival_Days",ascending=False))
print(workflow[["Patient_ID","Age","Survival_Days","Survival_Years","Expression_Log"]])
print(patients.sample(n=3,random_state=42))


# 17. Performance and Memory Optimization
For large datasets, memory usage matters. Prefer vectorized operations and consider categorical dtype for repeated strings.


In [ ]:
print("Original memory:",patients.memory_usage(deep=True).sum(),"bytes")
optimized=patients.copy()
for c in ["Gender","Tumor_Type","Treatment"]: optimized[c]=optimized[c].astype("category")
print("Optimized memory:",optimized.memory_usage(deep=True).sum(),"bytes")


# 18. Useful Extra Functions
A few small functions appear frequently in real workflows.


In [ ]:
s=pd.Series([-5,-2,0,4,8])
print(s.abs())
print(patients.sort_values(by=["Tumor_Type","Survival_Days"],ascending=[True,False])[["Patient_ID","Tumor_Type","Survival_Days"]])


# 19. Mini Practice Project — Clinical + Molecular Data
This is a small educational project, not the final real-world project. It combines cleaning, feature engineering, grouping, merging, reshaping, correlation, and ML preparation.


In [ ]:
clinical_data=pd.DataFrame({"Patient_ID":["P001","P002","P003","P004","P005","P006"],"Age":[45,62,51,70,38,62],"Gender":["Female","Male","Female","Male","Female","Male"],"Tumor_Type":["GBM","GBM","LGG","GBM","LGG","GBM"],"Treatment":["Surgery","Chemo","Radiation","Surgery","Radiation","Chemo"],"Survival_Days":[520,310,780,240,920,410]})
molecular_data=pd.DataFrame({"Patient_ID":["P001","P002","P003","P004","P005","P006"],"TP53":[8.2,6.5,4.1,9.0,3.8,7.2],"EGFR":[4.5,8.1,5.0,7.9,3.2,6.8],"IDH1":[2.1,3.5,7.8,1.9,8.2,3.1]})
project=pd.merge(clinical_data,molecular_data,on="Patient_ID",how="inner")
project["Survival_Years"]=project["Survival_Days"]/365.25
project["Age_Group"]=pd.cut(project["Age"],bins=[0,49,64,100],labels=["Young","Middle","Older"])
print(project)


In [ ]:
print(project.groupby("Tumor_Type").agg({"Age":"mean","Survival_Days":["mean","median"],"TP53":"mean","EGFR":"mean","IDH1":"mean"}))
print(project[["Age","Survival_Days","TP53","EGFR","IDH1"]].corr())


In [ ]:
X=project[["Age","TP53","EGFR","IDH1"]].copy()
y=project["Survival_Days"].copy()
print("X:
",X)
print("y:
",y)


# 20. Exercises
Try these before looking at the notebook examples again:

1. Count unique tumor types.
2. Count patients per treatment.
3. Calculate mean survival per tumor type.
4. Filter ages between 40 and 60.
5. Find the three highest TP53 values.
6. Create `Survival_Years`.
7. Create Young/Middle/Older age groups.
8. Merge clinical and molecular data.
9. Create a Tumor_Type × Treatment crosstab.
10. Calculate a correlation matrix.
11. Convert categorical columns to `category`.
12. One-hot encode categorical variables.
13. Use `groupby()` + `transform()` to calculate mean IDH1 per tumor type.
14. Convert molecular data to long format with `melt()`.
15. Find the two patients with highest survival using `nlargest()`.


# 21. Common Mistakes
**1.** `agg()` reduces groups while `transform()` preserves row alignment.

**2.** `pivot()` requires unique index/column combinations; use `pivot_table()` when aggregation is needed.

**3.** Correlation is not causation.

**4.** Arbitrary integer encoding can imply a false order; one-hot encoding is often better for nominal categories.

**5.** Do not automatically fill or drop missing values without understanding why they are missing.

**6.** Avoid `apply()` when a clear vectorized operation exists.

**7.** Prevent data leakage when preparing ML data.


# 22. Pandas Part 2 Cheat Sheet
| Task | Function |
|---|---|
| Count values | `value_counts()` |
| Unique values | `unique()` / `nunique()` |
| Group | `groupby()` |
| Aggregate | `agg()` |
| Group-aligned result | `transform()` |
| Custom function | `apply()` |
| Map values | `map()` |
| Membership | `isin()` |
| Range | `between()` |
| Query | `query()` |
| Conditional | `where()` / `mask()` |
| Replace | `replace()` |
| Pipeline columns | `assign()` |
| Limit values | `clip()` |
| Ranking/top N | `rank()` / `nlargest()` / `nsmallest()` |
| Missing values | `dropna()` / `fillna()` / `ffill()` / `bfill()` / `interpolate()` |
| Combine missing data | `combine_first()` |
| Duplicates | `duplicated()` / `drop_duplicates()` |
| Join tables | `merge()` / `join()` / `concat()` |
| Reshape | `melt()` / `pivot()` / `pivot_table()` |
| Frequency table | `crosstab()` |
| Expand lists | `explode()` |
| Binning | `cut()` / `qcut()` |
| Statistics | `corr()` / `cov()` |
| Sequential | `shift()` / `diff()` / `pct_change()` |
| Windows | `rolling()` / `expanding()` |
| Dates | `to_datetime()` / `.dt` |
| Index | `set_index()` / `reset_index()` |
| ML encoding | `get_dummies()` / `factorize()` |
| Select dtypes | `select_dtypes()` |
| Strings | `.str.contains()` / `.str.extract()` / `.str.split()` / `.str.replace()` |
| Memory | `memory_usage()` |


# 23. Golden Rules
1. Inspect before manipulating.
2. Use `groupby()` for group questions.
3. Use `agg()` for summaries and `transform()` for aligned features.
4. Use `merge()` for relational tables and `concat()` for stacking.
5. Use `melt()`/`pivot()` to reshape.
6. Investigate missingness before handling it.
7. Correlation does not prove causation.
8. Prefer vectorization when practical.
9. Prevent data leakage.
10. Keep raw and processed data separate.


# 24. Part 1 vs Part 2
**Part 1 — Foundations:** Series, DataFrames, reading files, inspection, selection, basic filtering/statistics, missing values, dtypes, renaming, deletion, duplicates, basic strings.

**Part 2 — Intermediate & Advanced:** GroupBy, aggregation, transformation, advanced filtering, merging/joining, reshaping, categorical data, time series, statistics, feature engineering, ML preparation, and performance.

> **Part 1 teaches you how to work with a DataFrame. Part 2 teaches you how to manipulate and analyze data at a much deeper level.**


# 25. What Comes Next?
After Parts 1 and 2, the next step can be a dedicated **real-world Pandas project** based on a biological or biomedical dataset. The final project can combine cleaning → filtering → GroupBy → feature engineering → merging → reshaping → statistical analysis → visualization → ML preparation.
